In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
MedSAM Echo Image Segmentation Tool

This script provides a comprehensive implementation for using MedSAM to segment
echo (ultrasound) images, with support for both local and remote execution.

Features:
- Load and preprocess echo images
- Interactive bounding box annotation
- Run MedSAM inference locally or on remote servers
- Visualize and save segmentation results

Requirements:
- PyTorch
- OpenCV
- Matplotlib
- segment_anything (MedSAM)
"""

import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
import cv2
import argparse
import time
from pathlib import Path
import json
import socket
from typing import List, Dict, Tuple, Optional, Union

# Check if running in Jupyter or as standalone script
IN_JUPYTER = 'ipykernel' in sys.modules
if IN_JUPYTER:
    from IPython.display import display, clear_output
    # Enable interactive matplotlib in Jupyter
    from matplotlib import rcParams
    rcParams['figure.figsize'] = (12, 8)

    try:
        get_ipython().run_line_magic('matplotlib', 'widget')
    except:
        print("Warning: Could not enable widget backend for matplotlib")


class MedSAMModel:
    """MedSAM model wrapper for medical image segmentation"""

    def __init__(self, checkpoint_path: str, device: str = None):
        """
        Initialize the MedSAM model.

        Args:
            checkpoint_path: Path to the MedSAM model checkpoint
            device: Device to use (cpu or cuda:0)
        """
        self.checkpoint_path = checkpoint_path
        if device is None:
            self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device

        self.model = None
        self.load_model()

    def load_model(self):
        """Load the MedSAM model"""
        try:
            # Import MedSAM components
            from segment_anything import sam_model_registry

            print(f"Loading MedSAM model from: {self.checkpoint_path}")
            print(f"Using device: {self.device}")

            # Load the model
            self.model = sam_model_registry['vit_b'](checkpoint=self.checkpoint_path)
            self.model = self.model.to(self.device)
            self.model.eval()

            print("MedSAM model loaded successfully")
        except Exception as e:
            print(f"Error loading MedSAM model: {e}")
            raise

    def preprocess_image(self, image: np.ndarray) -> torch.Tensor:
        """
        Preprocess the image for MedSAM.

        Args:
            image: Input image (RGB format)

        Returns:
            Preprocessed image tensor
        """
        # Ensure RGB format
        if len(image.shape) == 2:  # Convert grayscale to RGB
            image = np.stack([image] * 3, axis=2)
        elif image.shape[2] == 1:  # Convert single channel to RGB
            image = np.repeat(image, 3, axis=2)

        # Convert to tensor and normalize
        image_tensor = torch.from_numpy(image).float().permute(2, 0, 1).unsqueeze(0)

        # Normally there would be normalization here, but MedSAM handles this internally

        return image_tensor.to(self.device)

    def segment_with_box(self, image: np.ndarray, box: np.ndarray,
                         multimask_output: bool = False) -> Tuple[np.ndarray, np.ndarray]:
        """
        Segment the image using a bounding box prompt.

        Args:
            image: Input image (RGB format)
            box: Bounding box coordinates [x_min, y_min, x_max, y_max]
            multimask_output: Whether to return multiple masks

        Returns:
            Tuple of (masks, scores)
        """
        with torch.no_grad():
            # Preprocess image
            image_tensor = self.preprocess_image(image)

            # Prepare box
            box_torch = torch.tensor(box, dtype=torch.float32).unsqueeze(0).to(self.device)

            # Create embedding
            image_embedding = self.model.image_encoder(image_tensor)

            # Prepare prompt
            sparse_embeddings, dense_embeddings = self.model.prompt_encoder(
                points=None,
                boxes=box_torch,
                masks=None,
            )

            # Decode masks
            mask_predictions, scores = self.model.mask_decoder(
                image_embeddings=image_embedding,
                image_pe=self.model.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_embeddings,
                dense_prompt_embeddings=dense_embeddings,
                multimask_output=multimask_output,
            )

            # Convert to numpy
            masks = mask_predictions.cpu().numpy()
            scores = scores.cpu().numpy()

            return masks, scores


class EchoImageProcessor:
    """Class for processing echo images"""

    def __init__(self, medsam_model: MedSAMModel):
        """
        Initialize the echo image processor.

        Args:
            medsam_model: MedSAM model wrapper
        """
        self.medsam_model = medsam_model
        self.current_image = None
        self.current_mask = None
        self.current_score = None

    def load_image(self, image_path: str) -> np.ndarray:
        """
        Load an echo image from file.

        Args:
            image_path: Path to the image

        Returns:
            Loaded image
        """
        # Load image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        self.current_image = image
        return image

    def segment_with_box(self, box: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Segment the current image using a bounding box.

        Args:
            box: Bounding box coordinates [x_min, y_min, x_max, y_max]

        Returns:
            Tuple of (mask, score)
        """
        if self.current_image is None:
            raise ValueError("No image loaded")

        masks, scores = self.medsam_model.segment_with_box(
            self.current_image, box, multimask_output=False)

        self.current_mask = masks[0, 0]  # First mask
        self.current_score = scores[0, 0]  # First score

        return self.current_mask, self.current_score

    def save_results(self, output_dir: str, filename: str):
        """
        Save the segmentation results.

        Args:
            output_dir: Output directory
            filename: Base filename
        """
        if self.current_image is None or self.current_mask is None:
            raise ValueError("No segmentation results to save")

        os.makedirs(output_dir, exist_ok=True)

        # Save mask
        mask_path = os.path.join(output_dir, f"{filename}_mask.png")
        cv2.imwrite(mask_path, (self.current_mask * 255).astype(np.uint8))

        # Save visualization
        vis_path = os.path.join(output_dir, f"{filename}_vis.png")
        vis_img = self.create_visualization()
        cv2.imwrite(vis_path, cv2.cvtColor(vis_img, cv2.COLOR_RGB2BGR))

        # Save metadata
        meta_path = os.path.join(output_dir, f"{filename}_meta.json")
        metadata = {
            "score": float(self.current_score),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        with open(meta_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        print(f"Results saved to {output_dir}")

    def create_visualization(self) -> np.ndarray:
        """
        Create a visualization of the segmentation result.

        Returns:
            Visualization image
        """
        if self.current_image is None or self.current_mask is None:
            raise ValueError("No segmentation results to visualize")

        # Create a copy of the image
        vis_img = self.current_image.copy()

        # Create mask overlay
        mask_color = np.zeros_like(vis_img)
        mask_color[:, :, 0] = 255  # Red channel
        mask_overlay = np.where(self.current_mask[:, :, np.newaxis] == 1, mask_color, 0)

        # Blend
        alpha = 0.5
        vis_img = cv2.addWeighted(vis_img, 1, mask_overlay, alpha, 0)

        # Add score text
        text = f"Score: {self.current_score:.3f}"
        cv2.putText(vis_img, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                   1, (255, 255, 255), 2, cv2.LINE_AA)

        return vis_img


class InteractiveAnnotator:
    """Interactive annotation tool for echo images"""

    def __init__(self, processor: EchoImageProcessor):
        """
        Initialize the interactive annotator.

        Args:
            processor: Echo image processor
        """
        self.processor = processor
        self.fig, self.axes = plt.subplots(1, 2, figsize=(12, 6))
        self.box_coordinates = None
        self.selector = None

    def setup(self, image: np.ndarray):
        """
        Setup the interactive annotation.

        Args:
            image: Input image
        """
        # Clear axes
        for ax in self.axes:
            ax.clear()

        # Display image
        self.axes[0].imshow(image)
        self.axes[0].set_title("Draw bounding box")
        self.axes[1].set_title("Segmentation result")

        # Setup selector
        self.selector = RectangleSelector(
            self.axes[0], self.onselect,
            drawtype='box',
            rectprops=dict(facecolor='red', edgecolor='black', alpha=0.2, fill=True),
            interactive=True
        )

        # Update layout
        plt.tight_layout()

    def onselect(self, eclick, erelease):
        """
        Handle selection event.

        Args:
            eclick: Event at click
            erelease: Event at release
        """
        x1, y1 = int(eclick.xdata), int(eclick.ydata)
        x2, y2 = int(erelease.xdata), int(erelease.ydata)
        self.box_coordinates = np.array([min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)])
        print(f"Selected box: {self.box_coordinates}")

        # Segment
        mask, score = self.processor.segment_with_box(self.box_coordinates)

        # Display result
        self.axes[1].clear()
        self.axes[1].imshow(self.processor.current_image)
        self.axes[1].imshow(mask, alpha=0.5, cmap='jet')
        self.axes[1].set_title(f"Segmentation (score: {score:.3f})")

        # Update figure
        self.fig.canvas.draw_idle()

    def show(self):
        """Show the interactive annotation tool"""
        plt.show()


class RemoteServer:
    """Class for remote server functionality"""

    def __init__(self, host: str, port: int = 22):
        """
        Initialize the remote server connection.

        Args:
            host: Remote host
            port: SSH port
        """
        self.host = host
        self.port = port
        self.connected = False

    def connect(self, username: str, password: str = None, key_path: str = None):
        """
        Connect to the remote server.

        Args:
            username: SSH username
            password: SSH password
            key_path: Path to SSH key
        """
        try:
            import paramiko

            # Setup client
            client = paramiko.SSHClient()
            client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

            # Connect
            if key_path:
                client.connect(self.host, self.port, username, key_filename=key_path)
            else:
                client.connect(self.host, self.port, username, password)

            self.client = client
            self.connected = True
            print(f"Connected to {self.host}")

        except ImportError:
            print("Paramiko is required for SSH connections. Install with: pip install paramiko")
            raise
        except Exception as e:
            print(f"Failed to connect to {self.host}: {e}")
            raise

    def upload_file(self, local_path: str, remote_path: str):
        """
        Upload a file to the remote server.

        Args:
            local_path: Local file path
            remote_path: Remote file path
        """
        if not self.connected:
            raise ValueError("Not connected to remote server")

        try:
            sftp = self.client.open_sftp()
            sftp.put(local_path, remote_path)
            sftp.close()
            print(f"Uploaded {local_path} to {remote_path}")
        except Exception as e:
            print(f"Failed to upload file: {e}")
            raise

    def download_file(self, remote_path: str, local_path: str):
        """
        Download a file from the remote server.

        Args:
            remote_path: Remote file path
            local_path: Local file path
        """
        if not self.connected:
            raise ValueError("Not connected to remote server")

        try:
            sftp = self.client.open_sftp()
            sftp.get(remote_path, local_path)
            sftp.close()
            print(f"Downloaded {remote_path} to {local_path}")
        except Exception as e:
            print(f"Failed to download file: {e}")
            raise

    def execute_command(self, command: str) -> Tuple[str, str]:
        """
        Execute a command on the remote server.

        Args:
            command: Command to execute

        Returns:
            Tuple of (stdout, stderr)
        """
        if not self.connected:
            raise ValueError("Not connected to remote server")

        try:
            stdin, stdout, stderr = self.client.exec_command(command)
            stdout_str = stdout.read().decode()
            stderr_str = stderr.read().decode()
            return stdout_str, stderr_str
        except Exception as e:
            print(f"Failed to execute command: {e}")
            raise

    def disconnect(self):
        """Disconnect from the remote server"""
        if self.connected:
            self.client.close()
            self.connected = False
            print(f"Disconnected from {self.host}")


def main():
    """Main function"""
    # Parse arguments
    parser = argparse.ArgumentParser(description="MedSAM Echo Image Segmentation")
    parser.add_argument("--checkpoint", type=str, required=True, help="Path to MedSAM checkpoint")
    parser.add_argument("--image", type=str, required=True, help="Path to echo image")
    parser.add_argument("--output", type=str, default="output", help="Output directory")
    parser.add_argument("--remote", action="store_true", help="Use remote server")
    parser.add_argument("--host", type=str, default=None, help="Remote host")
    parser.add_argument("--username", type=str, default=None, help="SSH username")
    parser.add_argument("--interactive", action="store_true", help="Use interactive annotation")
    args = parser.parse_args()

    # Check if running in interactive mode
    if not args.interactive and IN_JUPYTER:
        args.interactive = True

    # Initialize model
    model = MedSAMModel(args.checkpoint)

    # Initialize processor
    processor = EchoImageProcessor(model)

    # Load image
    image = processor.load_image(args.image)
    print(f"Loaded image with shape: {image.shape}")

    if args.remote:
        # Remote server setup would go here
        # This is a placeholder for the remote server functionality
        print("Remote server functionality not implemented in this demo")
        return

    if args.interactive:
        # Interactive annotation
        annotator = InteractiveAnnotator(processor)
        annotator.setup(image)
        annotator.show()
    else:
        # Automatic segmentation with center box
        h, w = image.shape[:2]
        # Create a center box covering 50% of the image
        center_box = np.array([w//4, h//4, 3*w//4, 3*h//4])
        mask, score = processor.segment_with_box(center_box)

        # Save results
        processor.save_results(args.output, Path(args.image).stem)

        # Display result
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(image)
        plt.title("Original Image")

        plt.subplot(1, 2, 2)
        plt.imshow(image)
        plt.imshow(mask, alpha=0.5, cmap='jet')
        plt.title(f"Segmentation (score: {score:.3f})")

        plt.tight_layout()
        plt.show()


if __name__ == "__main__":
    print(f"Running on: {socket.gethostname()}")
    main()